<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Operating-Systems/04-cpu-scheduling-and-dispatch.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Operating Systems guideline](Operating-Systems.html)


## **CPU Scheduling and Dispatch**

At any instant, a computer may contain far more runnable threads than hardware execution contexts. **CPU scheduling** is the operating system decision process that maps those runnable threads onto processors over time. It answers three linked questions: which thread should run next, on which CPU should it run, and when should the current thread stop running? **Dispatch** is the mechanism that carries out the answer by changing the processor from one execution context to another.

Scheduling matters even when every program is correct. A poor policy can leave an interactive terminal waiting behind a long computation, move a thread so often that its cache state is repeatedly lost, give one user much more CPU time than another, or complete a real-time computation after its result has become useless. No single policy is best for all of these goals. A scheduler is therefore an explicit statement about what the system values and what it assumes about future work.

The recurring pipeline provides a useful concrete case:

```bash
cat input.txt | grep kernel > result.txt
```

`cat` is runnable while it copies input, then may block while storage supplies another page. `grep` may block while the pipe is empty, become runnable when `cat` writes data, consume CPU while matching lines, and block again when output storage applies backpressure. The shell normally sleeps while waiting for both children. Scheduling does not create this producer-consumer structure, but it determines how quickly each runnable stage receives a CPU and whether one stage is delayed enough to leave the other idle.

### **The CPU Scheduling Problem**

A thread moves among states such as **running**, **runnable**, and **blocked**. A runnable thread has everything it needs except a processor. The scheduler operates primarily on this runnable set. It is invoked, or a scheduling decision is requested, after events such as:

- the running thread blocks, exits, or voluntarily yields;
- a timer interrupt indicates that a time allocation has expired;
- an I/O completion or synchronization event wakes another thread;
- a higher-urgency task becomes runnable;
- a CPU becomes idle or the system detects load imbalance;
- a task changes priority, affinity, deadline, or resource-control group.

![Runnable threads enter a run queue, scheduling policy selects one, and the dispatcher makes that selection execute.](assets/scheduler-policy-dispatch-loop.svg){fig-alt="Flowchart separating scheduling policy from dispatch mechanism and showing block, preempt, and exit paths from the CPU." width="94%"}

*Figure: original explanatory diagram created for this chapter. The event and context-switch structure follows the scheduler model in the [xv6 book](https://pdos.csail.mit.edu/6.1810/2025/xv6/book-riscv-rev5.pdf) and the Linux policy terminology in [`sched(7)`](https://man7.org/linux/man-pages/man7/sched.7.html).*

The scheduler does not normally know a thread's exact future. It can observe prior CPU use, wakeup history, priorities, weights, deadlines supplied by an application, and hardware topology. It must turn incomplete evidence into a decision quickly because scheduling code runs frequently and directly adds overhead. This creates the central tension: sophisticated policy may make better decisions, but the decision itself must remain bounded, scalable, and safe inside the kernel.

On one CPU, scheduling is an ordering problem. On several CPUs, it is simultaneously an ordering and placement problem. A thread can be selected promptly yet placed badly: moving it to an idle remote CPU may start it sooner but discard warm cache state or separate it from its memory on a NUMA machine. Modern scheduling therefore combines temporal policy, spatial placement, and resource accounting.

#### **Policy Versus Dispatch Mechanism**

**Policy** expresses preference. FCFS prefers the earliest arrival, SJF prefers the shortest predicted job, Round Robin rotates among peers, a fair scheduler tries to match CPU service to weights, and EDF prefers the earliest absolute deadline. **Mechanism** provides reusable operations: maintain runnable queues, enter the kernel, stop a thread, save and restore context, change address-space state when required, program a timer, and return to user mode.

Keeping them conceptually separate makes systems easier to reason about. The same low-level context-switch mechanism can serve fair, priority, and real-time classes. Conversely, changing the queue order does not require redesigning how registers are saved. A simplified interface looks like this:

```text
on_scheduling_event(event):
    update_accounting(current, event.time)
    update_runnable_set(event)

    next = policy.pick_next(runnable_set)
    if next != current:
        dispatcher.switch(current, next)
    else:
        return_to(current)
```

The distinction prevents two common errors. First, a timer interrupt is not itself a context switch; the kernel may decide that the interrupted thread should continue. Second, a context switch is not necessarily a process switch. Switching between two threads in one process changes stacks and registers but can retain the same address space, whereas switching processes may also change page-table context and affect the TLB.

A useful abstract run-queue interface is:

| Operation | Meaning | Policy-dependent question |
|---|---|---|
| `enqueue(thread)` | make a thread eligible to run | where and at what rank? |
| `dequeue(thread)` | remove a blocked, exiting, or migrated thread | how is partial service preserved? |
| `pick_next()` | select an eligible thread | earliest arrival, minimum runtime, highest priority, or deadline? |
| `account(thread, delta)` | charge consumed CPU time | wall time, weighted virtual time, budget, or quantum? |
| `should_preempt(current, new)` | compare current work with a new event | does urgency justify switch cost? |

The asymptotic cost of these operations matters, but constants and contention matter too. A FIFO queue can offer constant-time insertion and removal. A policy ordered by remaining time, deadline, or virtual runtime often uses a heap or balanced tree and pays `O(log n)` updates. A single global queue is easy to balance but can become a shared lock bottleneck; per-CPU queues scale better but require migration and balancing logic.

#### **Workload Models and Scheduling Assumptions**

Scheduling analysis starts by stating what a **job** represents. In a simple model, job $i$ has:

- release or arrival time $r_i$;
- required CPU service, often called burst time, $p_i$;
- first execution time $S_i$ and completion time $C_i$;
- optional deadline $d_i$, static priority $q_i$, or weight $w_i$;
- one CPU and no internal parallelism.

Real threads are more complicated. They alternate between CPU bursts and blocking operations, their burst lengths are initially unknown, wakeups can depend on other threads, and execution time changes with cache state, frequency scaling, and interference. A model is still useful if its assumptions are kept visible.

This chapter uses one small workload repeatedly:

| Job | Arrival $r_i$ | CPU burst $p_i$ | Informal role |
|---|---:|---:|---|
| A | 0 | 8 | long CPU-bound work |
| B | 1 | 4 | medium request |
| C | 2 | 2 | short request |
| D | 4 | 1 | very short interactive work |

The table deliberately exposes uncertainty. FCFS needs only arrival order. SJF and SRTF appear to know exact burst lengths, which makes them **clairvoyant** textbook policies. A real scheduler must estimate those lengths or infer interactivity from behavior. Round Robin avoids burst prediction but introduces a quantum. Priority policies need a trusted method for assigning priority. Real-time policies require meaningful timing parameters and admission control.

Other assumptions can change a theorem or reverse a comparison:

| Assumption | Simplification | What changes when it is false |
|---|---|---|
| one processor | only one job executes at once | placement, migration, and parallel execution appear |
| all jobs arrive together | no future arrivals interrupt the plan | online decisions and preemption become important |
| exact burst lengths known | SJF/SRTF can rank perfectly | prediction error can delay the wrong job |
| zero switch cost | arbitrarily small quanta seem attractive | overhead and cache disruption limit preemption rate |
| independent jobs | order only affects metrics | locks, pipelines, and priority inversion create dependencies |
| stationary workload | recent behavior predicts future behavior | phase changes require feedback and adaptation |

Always ask whether a result is **offline** or **online**, **preemptive** or **non-preemptive**, and whether it optimizes a mean, a tail, a fairness target, or a deadline constraint. "Shortest is optimal" is meaningful only after those qualifications.

### **Scheduling Goals and Metrics**

Scheduling objectives often conflict. Batch processing may value total completions per hour. An editor values low response latency. A shared server values isolation and proportional allocation. Audio processing values bounded jitter, while a flight controller may require a provable deadline. A useful evaluation reports several metrics and the workload distribution instead of declaring a policy "fast."

#### **Turnaround, Response, Waiting, and Throughput**

For job $i$, define:

$$
\begin{aligned}
\text{turnaround}_i &= C_i-r_i,\\
\text{response}_i &= S_i-r_i,\\
\text{waiting}_i &= \text{time runnable but not running}.
\end{aligned}
$$

If the simple model contains only runnable and running time after arrival, then:

$$
\text{waiting}_i = (C_i-r_i)-p_i.
$$

That identity does not hold unchanged when a real thread spends time blocked for I/O or synchronization; blocked time is part of turnaround but is not ready-queue waiting. Measurement tools must distinguish these states.

![Timeline distinguishing arrival, first execution, preemption, completion, response time, turnaround time, waiting time, and CPU service.](assets/scheduling-metrics-timeline.svg){fig-alt="Timeline for one preempted job showing how scheduling metrics are calculated." width="92%"}

*Figure: original explanatory diagram created for this chapter from the standard timing terms used by [`sched(7)`](https://man7.org/linux/man-pages/man7/sched.7.html).*

**Throughput** is the number of completed jobs per unit time. **CPU utilization** is the fraction of an observation interval during which the processor performs useful scheduled work. High utilization is not sufficient evidence of good service: an overloaded system can keep every CPU busy while queues and tail latency grow without bound.

**Slowdown**, also called normalized turnaround, compares completion delay with the job's own service demand:

$$
\text{slowdown}_i = \frac{C_i-r_i}{p_i}.
$$

A turnaround of ten units is minor for a job needing nine units and severe for a job needing one. Slowdown makes that asymmetry visible, although extremely short jobs may require a lower-bound convention to prevent unstable ratios.

Means should be accompanied by distributions. Average response can improve while the 99th percentile becomes much worse, and two policies with identical mean waiting time can feel very different if one is predictable and the other occasionally stalls a request for seconds. For interactive and service workloads, report at least median and high-percentile response or queueing delay. For batch workloads, report throughput, turnaround, and bounded starvation behavior.

#### **Fairness, Predictability, and Deadline Satisfaction**

Fairness has no universal definition. Equal CPU time among runnable threads is one policy. Weighted fairness gives thread $i$ a target share

$$
s_i = \frac{w_i}{\sum_{j \in R} w_j},
$$

where $R$ is the runnable set. Per-user or per-container fairness may first divide service among groups and only then among their threads. Without this hierarchy, a user can gain a larger share merely by creating more threads.

For observed allocations $x_1,\ldots,x_n$, Jain's index is one summary:

$$
J(x_1,\ldots,x_n)=\frac{(\sum_i x_i)^2}{n\sum_i x_i^2}.
$$

It ranges from $1/n$ to $1$, but a high value only means allocations are similar. It does not prove that the intended weights, priorities, or service-level objectives were respected.

**Predictability** concerns variation: response-time variance, scheduling jitter, and the maximum interval for which an eligible thread can be delayed. A policy with slightly worse average latency but a tight upper tail may be better for media and control loops. **Deadline satisfaction** asks whether $C_i \le d_i$. Useful measurements include miss ratio, maximum lateness $\max(C_i-d_i)$, and tardiness $\max(0,C_i-d_i)$.

These objectives create explicit trade-offs:

- minimizing mean waiting favors short work and can starve long jobs;
- frequent rotation improves initial response but spends more time switching;
- strict priority protects urgent work but can deny service to lower levels;
- preserving affinity protects locality but can leave another CPU idle;
- maximizing throughput under sustained load can increase latency and jitter;
- meeting hard deadlines requires admission and bounded interference, not merely high average speed.

### **Non-Preemptive Scheduling**

A non-preemptive scheduler lets the selected job retain the CPU until it blocks, exits, or yields. The approach reduces policy intervention and preserves locality during a burst, but newly arrived urgent work cannot displace the current job. Non-preemptive policies are therefore easiest to analyze when jobs are short, trusted, or naturally yield at bounded intervals.

#### **First-Come, First-Served**

**First-Come, First-Served** (FCFS), also called FIFO scheduling, runs ready jobs in arrival order. It needs no burst prediction and can be implemented with a queue:

```text
enqueue(job):
    ready.push_back(job)

pick_next():
    return ready.pop_front()
```

Both operations are `O(1)` with a linked queue or circular buffer. Ties require a stable rule, such as insertion order or job identifier. Once a job starts its CPU burst, later arrivals wait.

For the common workload, FCFS produces:

```text
time  0--------8----12--14-15
CPU   |   A    | B  | C |D|
```

The waiting times are `A=0`, `B=7`, `C=10`, and `D=10`, so average waiting is `6.75`. The result demonstrates the **convoy effect**: short jobs C and D wait behind long job A. In an I/O-heavy system, a CPU-bound job can similarly delay several I/O-oriented jobs, causing devices to become idle while the CPU queue drains and then causing a burst of device requests later.

FCFS remains useful when ordering itself matters, jobs are similar in size, preemption is impossible or expensive, or it forms the queue discipline inside a larger policy. Its simplicity also makes it a valuable baseline. FCFS prevents starvation only when every earlier burst is finite; arrival order alone cannot bound response behind a thread that may run indefinitely.

#### **Shortest Job First**

**Shortest Job First** (SJF) selects the available job with the smallest predicted CPU burst and then runs it non-preemptively. When all considered jobs are simultaneously available, service times are known, jobs are independent, and the objective is mean waiting time, ordering them from shortest to longest minimizes the sum of waiting times. An exchange argument explains why: if adjacent jobs have lengths $a>b$ but run as $a$ then $b$, swapping them reduces the second job's wait by $a-b$ without delaying earlier jobs. With staggered releases, choosing the shortest currently ready job is an online rule; a future arrival cannot change a non-preemptive decision already made, and allowing intentional idle time produces a different optimization problem.

```text
pick_next(ready):
    return job with minimum (predicted_burst, arrival_time, id)
```

A binary heap gives `O(log n)` enqueue and removal, but a plain scan can be reasonable for tiny queues. SJF does not reconsider the running job. In the example, only A exists at time 0, so A still runs to completion. At time 8, the scheduler chooses D, then C, then B:

```text
time  0--------8-9--11----15
CPU   |   A    |D| C | B  |
```

Average waiting falls from `6.75` to `5.25`, but D still waited four units because SJF could not preempt A. This is an important boundary: choosing optimally at a decision point cannot recover an opportunity that a non-preemptive policy never creates.

Real systems do not know the next burst. One classical estimate applies exponential smoothing to prior bursts:

$$
\tau_{n+1}=\alpha t_n+(1-\alpha)\tau_n, \qquad 0\le\alpha\le1,
$$

where $t_n$ is the observed burst and $\tau_n$ is the prior prediction. A larger $\alpha$ reacts quickly but is noisy; a smaller value is stable but slow to detect phase changes. Prediction errors can put a supposedly short job at the front for a long time, and a steady stream of short arrivals can starve long work. Aging, service guarantees, or a feedback scheduler is needed if bounded progress matters.

### **Preemptive Scheduling**

A preemptive scheduler may stop a runnable thread before it blocks or completes. Preemption lets the system react to short arrivals, expiring quanta, priorities, and deadlines. It also creates overhead: kernel entry, accounting, queue updates, register switching, cache disruption, and sometimes TLB effects. The right question is therefore not whether preemption is good, but which event justifies it and how often it may occur.

![FCFS, SJF, SRTF, and Round Robin applied to the same arrivals and CPU bursts.](assets/classic-scheduling-comparison-animated.svg){fig-alt="Four aligned Gantt charts compare classic scheduling algorithms with a moving simulated-time cursor." width="100%"}

*Figure: original animated comparison created for this chapter. The complete schedules remain visible when reduced-motion preferences disable animation.*

The aligned chart is more informative than four unrelated examples. It shows that the algorithms optimize different experiences: SRTF completes new short jobs rapidly, Round Robin starts multiple jobs early, and the two non-preemptive policies cannot react while A is running.

#### **Shortest Remaining Time First**

**Shortest Remaining Time First** (SRTF) is the preemptive form of SJF. At every arrival or completion, it runs the job with the least remaining service. If a new job is shorter than the current job's remainder, it preempts immediately.

```text
on arrival(job):
    ready.insert(job, key=remaining_time)
    if job.remaining < current.remaining:
        preempt(current)

pick_next():
    return ready.remove_min()
```

For the common workload, A runs for one unit. B arrives with length 4 while A has 7 remaining, so B preempts A. C then arrives with length 2 while B has 3 remaining. C completes at time 4, D completes at 5, B completes at 8, and A completes at 15. Average waiting is `2.50`.

Under ideal single-CPU assumptions with known processing times, SRTF minimizes mean flow time, where flow time is turnaround. The intuition is that finishing a short job removes one member from the waiting population sooner. The result does not mean SRTF is universally optimal: it can starve long work, is highly sensitive to prediction error, and may switch frequently when estimates are close.

A min-heap supports arrivals and selections in `O(log n)`. Remaining-time accounting itself is constant time, but every preemption also pays a machine cost. Practical variants introduce thresholds, minimum granularity, or fairness accounting so a tiny predicted advantage does not trigger an expensive switch.

#### **Round Robin**

**Round Robin** (RR) gives each runnable peer at most a **time quantum** $q$. A thread that blocks or finishes before $q$ leaves the CPU early. A thread that consumes the whole quantum is appended to the queue tail.

```text
ready.push_back(new_job)

while ready is not empty:
    job = ready.pop_front()
    run job for min(q, job.remaining)
    enqueue arrivals that occurred during this interval
    if job still has work:
        ready.push_back(job)
```

With `q=2` and arrivals inserted before the expired job is requeued at the same timestamp, the example runs:

```text
A 0-2, B 2-4, C 4-6, A 6-8,
D 8-9, B 9-11, A 11-13, A 13-15
```

Average waiting is `4.75`, worse than SRTF, but B, C, and D receive first service before FCFS would have completed A. Average response is `1.75` instead of FCFS's `6.75`. Exact tie behavior at a quantum boundary is part of the specification; a different rule can change individual response times without changing the core policy.

Quantum size controls the trade-off:

- as $q \rightarrow \infty$, RR approaches FCFS;
- a smaller $q$ reduces the maximum wait between turns when $n$ peers remain runnable, roughly toward $(n-1)q$ before accounting for new arrivals and overhead;
- if $q$ is close to context-switch cost $s$, efficiency can be poor. In a worst-case rotation where every quantum causes a switch, the useful fraction is approximately $q/(q+s)$;
- a very small quantum can also damage cache locality even when direct switch time looks acceptable.

Round Robin is a mechanism for time sharing, not a complete definition of fairness. Threads that frequently block may receive less CPU because they do not use full quanta, and multiple users can receive unequal shares if one creates many more runnable threads. Hierarchical weights or group scheduling address that policy question.

#### **Priority Scheduling and Aging**

Priority scheduling runs the highest-priority eligible thread. Priorities may be static, derived from deadlines, changed by feedback, inherited from a resource owner, or combined with a fair policy within each level. A preemptive priority scheduler immediately displaces lower-priority work when a higher-priority thread wakes; a non-preemptive version waits for the current burst to end.

Strict priority is appropriate only when the ordering expresses a real service requirement. Otherwise, a continuous stream of high-priority arrivals can cause **starvation**. **Aging** repairs this by improving a waiting job's effective priority over time. If lower numeric values denote higher priority, one simple rule is:

$$
q_i^{\text{effective}}(t)=\max(q_{\min}, q_i-\lfloor W_i(t)/A\rfloor),
$$

where $W_i(t)$ is accumulated ready-queue waiting and $A$ is an aging interval. The exact units and reset rule belong to the policy contract.

![Aging gradually makes an old low-priority job competitive; an illustrative multilevel feedback queue uses demotion and periodic boosts.](assets/priority-aging-mlfq.svg){fig-alt="Two-panel diagram of priority aging and a three-level multilevel feedback queue." width="96%"}

*Figure: original explanatory diagram created for this chapter. The MLFQ rules are intentionally labeled illustrative because implementations differ.*

Aging prevents indefinite waiting only if the improvement rate eventually overcomes incoming priorities and the scheduler honors the resulting rank. It does not solve **priority inversion**, where a high-priority thread waits for a lock held by a lower-priority thread. Priority inheritance and ceiling protocols address that dependency and are treated with synchronization in the next chapter.

### **Multilevel Feedback Queue Scheduling**

**Multilevel Feedback Queue** (MLFQ) scheduling approximates the responsiveness of shortest-job policies without knowing burst lengths in advance. It maintains several priority queues and uses observed CPU consumption as feedback. Short or frequently blocking jobs tend to remain near the top; jobs that repeatedly consume full allocations move downward and receive longer quanta.

A defensible illustrative policy is:

```text
1. New jobs enter Q0, the highest-priority queue.
2. Always choose a job from the highest nonempty queue.
3. Use Round Robin within a queue.
4. Charge CPU use against a level-specific allotment.
5. Demote a job after it consumes that allotment, even across voluntary yields.
6. Periodically boost all runnable jobs to Q0.
```

If Q0 uses quantum 2, Q1 uses 4, and Q2 uses 8, an interactive job that runs briefly and blocks for input receives quick service after wakeup, while a long computation gradually moves to queues with less frequent but longer turns. Longer lower-level quanta reduce switching overhead for CPU-bound work.

Two details determine whether the design is robust. First, accounting must span yields. If a job can reset its allotment by yielding just before a quantum expires, it can game the policy and remain at high priority. Second, a periodic global boost or equivalent aging rule is needed so jobs in lower queues cannot starve when high-priority arrivals continue.

MLFQ is a family, not one algorithm. Systems vary in number of queues, quantum growth, whether I/O completion preserves or raises priority, how sleep time is treated, and how group weights interact with feedback. Its advantage is adaptive behavior; its cost is more parameters and less transparent guarantees. A workload can also change phase, so old CPU-bound history should not condemn a newly interactive phase forever.

### **Proportional-Share Scheduling**

Proportional-share scheduling asks a different question from shortest-job or strict-priority scheduling: over a sufficiently long interval, what fraction of a resource should each client receive? A weight or ticket count expresses a relative entitlement. If A, B, and C have weights `60`, `30`, and `10` and remain runnable, their targets are 60%, 30%, and 10% of CPU service.

Shares should be defined over the correct entity. Per-thread weights let an application gain service by spawning threads. Hierarchical scheduling can first allocate among users, containers, or control groups and then distribute each group's allocation internally. Quotas add an upper bound, while weights usually resolve contention rather than reserving idle capacity.

#### **Lottery and Stride Scheduling**

**Lottery scheduling** represents rights as tickets. Each allocation draws a uniformly random winning ticket, so client $i$ wins with probability

$$
P(i)=\frac{t_i}{\sum_j t_j}.
$$

```text
winner = random_integer(1, total_tickets)
for client in runnable_clients:
    winner -= client.tickets
    if winner <= 0:
        run client for one quantum
        break
```

The mechanism is simple and flexible: tickets can be transferred, grouped, or denominated through currencies. Its guarantee is statistical. Over $m$ independent lotteries, a client's allocation fluctuates around its target; short windows can be noticeably uneven. Efficient selection can use a tree of ticket sums rather than a linear scan.

**Stride scheduling** replaces randomness with deterministic virtual progress. Choose a large constant $K$, set

$$
\text{stride}_i=\frac{K}{t_i},
$$

and repeatedly run the client with the smallest `pass` value:

```text
client = remove_min_by_pass()
run client for one quantum
client.pass += client.stride
insert_by_pass(client)
```

More tickets produce a smaller stride, so the client becomes minimum more often. A heap gives `O(log n)` selection and reinsertion. Stride reduces short-term variance relative to a lottery, but dynamic joins, departures, changing tickets, overflow, and pass normalization must be handled so a new client neither monopolizes nor waits excessively.

![Lottery uses a random ticket draw, whereas stride repeatedly chooses the smallest pass value; both target the same long-run shares.](assets/lottery-stride.svg){fig-alt="Side-by-side comparison of ticket ranges and deterministic stride pass values." width="96%"}

*Figure: original explanatory diagram based on Waldspurger and Weihl's [Lottery Scheduling paper](https://www.usenix.org/publications/library/proceedings/osdi/full_papers/waldspurger.pdf) and Waldspurger's [Lottery and Stride Scheduling dissertation](https://waldspurger.org/carl/papers/phd-mit-tr667.pdf).*

Neither mechanism is a deadline scheduler. A 10% share does not say when the next quantum will occur, especially under a randomized lottery, and a task can miss a deadline while receiving the correct long-run amount. Service shares, latency bounds, and deadline guarantees are separate contracts.

### **Preemption, Timer Interrupts, and Dispatch**

Preemption requires the kernel to regain control. A hardware timer is commonly programmed to generate an interrupt, but timer frequency and scheduling quantum are not necessarily identical. A tick may update accounting without switching, and tickless operation can program the next relevant event directly instead of interrupting at a fixed rate.

![Timer interrupt, kernel accounting, scheduling decision, optional context switch, and return to user execution.](assets/timer-preemption-dispatch.svg){fig-alt="Flowchart showing the full path from a timer interrupt to either continuing the current thread or dispatching another." width="96%"}

*Figure: original explanatory diagram created for this chapter from the trap and scheduling mechanisms described in the [xv6 book](https://pdos.csail.mit.edu/6.1810/2025/xv6/book-riscv-rev5.pdf).*

A simplified preemption path is:

1. The CPU finishes an instruction boundary permitted by the architecture and vectors to an interrupt handler.
2. Hardware and entry code establish kernel privilege and preserve enough state to run safely.
3. The kernel acknowledges the timer, updates elapsed runtime, expires timers, and records whether rescheduling is required.
4. At a safe scheduling point, policy compares the current thread with eligible alternatives.
5. If the current thread remains best, the kernel restores it. Otherwise, it enqueues that thread if still runnable and selects another.
6. The switch path saves callee context, changes kernel stack and architecture state, changes memory context if required, restores the selected context, and returns.

The **dispatch latency** is the delay between a thread becoming the selected runnable choice and actually executing. It can include time spent in non-preemptible kernel regions, interrupt handling, scheduler work, and the context switch. **Interrupt latency** is the delay before an interrupt handler starts. **Context-switch time** is only the state-transition portion. Treating them as synonyms makes performance diagnosis imprecise.

Direct switch cost is only part of the overhead. A migrated task may refill private caches, branch predictors may lose history, and a process switch can disturb address-translation state. These indirect costs can exceed the instructions in the switch routine. This is why a scheduler may preserve the current thread when two choices are nearly equal and why minimum granularity exists in practical fair schedulers.

The kernel must also control where preemption is legal. Switching while low-level scheduler data is inconsistent would corrupt the system. Kernels use short protected regions, interrupt state, preemption counters, and strict lock ordering to delay a switch until invariants hold. That mechanism does not excuse long non-preemptible work: it contributes directly to worst-case latency.

### **Multiprocessor Scheduling**

With $m$ processors, up to $m$ runnable threads can execute simultaneously. A global ready queue makes idle CPUs easy to feed but becomes a shared synchronization and cache-coherence point. Per-CPU queues reduce contention and preserve locality, but the scheduler must decide where a waking thread enters and when imbalance justifies migration.

![Per-CPU run queues inside two NUMA nodes show the trade-off among balance, cache affinity, memory locality, and heterogeneous CPU capacity.](assets/multicore-scheduling-topology.svg){fig-alt="Two NUMA nodes with per-CPU run queues, caches, local memory, a task migration path, and one high-capacity core." width="98%"}

*Figure: original explanatory diagram based on Linux [scheduler domains](https://docs.kernel.org/scheduler/sched-domains.html), [NUMA](https://docs.kernel.org/mm/numa.html), and [energy-aware scheduling](https://docs.kernel.org/scheduler/sched-energy.html) documentation.*

Two broad placement styles are:

- **push balancing**, where a busy CPU or periodic balancer moves work away;
- **pull balancing**, where an idle CPU searches another queue for movable work.

Production systems combine wakeup placement, periodic balancing, idle balancing, affinity constraints, and topology-aware domains. The objective is not perfectly equal queue lengths. One CPU may contain a single heavy runnable thread while another has several light threads, and moving a task may cost more than waiting briefly.

#### **Affinity, Load Balancing, and Cache Locality**

**Processor affinity** is the preference or constraint that keeps a thread on a set of CPUs. Soft or natural affinity favors the previous CPU because its cache may still contain useful lines. Hard affinity restricts eligibility through a CPU mask. Linux exposes this through `sched_setaffinity(2)` and tools such as `taskset`.

<details>
<summary><strong>Linux: inspect and constrain CPU affinity</strong></summary>

```bash
# Show the CPUs on which this shell may run.
taskset -pc $$

# Launch a CPU-bound command on logical CPUs 0 and 2 only.
taskset --cpu-list 0,2 ./worker

# Show each thread, its last observed CPU, class, nice value, and state.
ps -eLo pid,tid,psr,cls,rtprio,ni,stat,comm --sort=pid,tid
```

The mask limits legal placement; it does not reserve those CPUs or guarantee immediate execution. Container cpusets, online CPUs, and other kernel restrictions are intersected with the requested mask.

</details>

<details>
<summary><strong>C: pin the calling Linux thread to one CPU</strong></summary>

```c
#define _GNU_SOURCE
#include <errno.h>
#include <sched.h>
#include <stdio.h>
#include <string.h>

int main(void) {
    cpu_set_t allowed;
    CPU_ZERO(&allowed);
    CPU_SET(0, &allowed);  // The calling thread may run only on CPU 0.

    if (sched_setaffinity(0, sizeof(allowed), &allowed) == -1) {
        fprintf(stderr, "sched_setaffinity: %s\n", strerror(errno));
        return 1;
    }

    CPU_ZERO(&allowed);
    if (sched_getaffinity(0, sizeof(allowed), &allowed) == -1) {
        fprintf(stderr, "sched_getaffinity: %s\n", strerror(errno));
        return 1;
    }

    printf("CPU 0 allowed: %s\n", CPU_ISSET(0, &allowed) ? "yes" : "no");
    return 0;
}
```

Compile on Linux with `cc -O2 -Wall affinity.c -o affinity`. The API controls the calling thread when `pid` is zero. A POSIX-threaded program can use `pthread_setaffinity_np` for a particular `pthread_t`.

</details>

Affinity is a hint or constraint, not a free optimization. Pinning can improve repeatability and cache warmth, but it can also strand work on a busy CPU while another is idle, overload one shared cache, or conflict with interrupts and container policy. Measure migrations, cache misses, and latency before fixing a mask permanently.

#### **NUMA and Heterogeneous Cores**

On a **non-uniform memory access** (NUMA) machine, memory access cost depends on the relationship between the executing CPU and the physical memory node containing a page. Migrating a thread without its working set can turn local accesses into remote accesses. Moving pages as well has a cost and may be wasteful if the thread soon moves again. Linux represents topology through scheduling domains and can use NUMA balancing to coordinate task and page placement.

Heterogeneous processors add unequal capacity and energy cost. A high-capacity core may finish latency-sensitive work faster but consume more power; an efficiency core may be preferable for background work. Linux Energy Aware Scheduling uses a platform energy model to compare candidate placements on supported asymmetric topologies. This is a multi-objective decision, not a simple "fast jobs on big cores" rule: thermal state, current utilization, frequency, affinity, and the cost of waking a power domain can change the answer.

Three principles remain useful:

- balance **capacity-normalized load**, not merely thread counts;
- preserve cache and NUMA locality until imbalance or urgency outweighs it;
- keep administrative constraints explicit, because an unschedulable affinity mask cannot be repaired by policy cleverness.

### **Real-Time Scheduling**

A real-time system is one in which correctness depends on **when** a result is produced as well as what the result contains. "Real-time" does not mean "usually fast." A hard real-time deadline must not be missed under the stated fault and workload model. A firm deadline makes late results worthless, while a soft real-time system tolerates occasional lateness with degraded quality.

A periodic or sporadic task is commonly described by execution budget $C_i$, relative deadline $D_i$, and period or minimum inter-arrival time $T_i$. Each release creates a job with an absolute deadline. Safe analysis needs a credible worst-case execution time, bounded blocking and interrupt interference, a defined CPU model, and an admission test. Average execution time cannot establish a hard guarantee.

#### **Rate Monotonic and Earliest Deadline First**

**Rate Monotonic Scheduling** (RMS) assigns fixed priorities by period: a shorter period receives higher priority. Under the classical Liu-Layland assumptions of independent preemptible periodic tasks on one CPU, deadlines equal periods, and negligible overhead, the utilization

$$
U=\sum_{i=1}^{n}\frac{C_i}{T_i}
$$

is guaranteed schedulable by RMS if

$$
U \le n(2^{1/n}-1).
$$

The bound approaches $\ln 2 \approx 0.693$ as $n$ grows. It is sufficient, not necessary: exceeding it means this quick test cannot guarantee the task set, not that every such task set fails. Harmonic periods can reach full utilization, and exact response-time analysis can accept task sets rejected by the bound.

**Earliest Deadline First** (EDF) uses dynamic priority: among ready jobs, run the one with the earliest absolute deadline. Under the same ideal single-CPU model with implicit deadlines, EDF can schedule any task set with $U\le1$. Its stronger utilization result does not make implementation free; deadline ordering, overload behavior, admission, and bounded kernel latency still matter.

![RMS misses the first T2 deadline for a high-utilization task set, while EDF completes the same job before its absolute deadline.](assets/rms-edf-timeline.svg){fig-alt="Two Gantt charts compare rate-monotonic and earliest-deadline-first schedules for periodic tasks." width="98%"}

*Figure: original worked example based on the model introduced by Liu and Layland in [Scheduling Algorithms for Multiprogramming in a Hard-Real-Time Environment](https://dl.acm.org/doi/10.1145/321738.321743).*

The example has `T1: C=2, D=T=5` and `T2: C=4, D=T=7`, giving $U\approx0.971$. RMS always gives T1 higher priority and preempts T2 at time 5, leaving one unit unfinished at T2's deadline 7. EDF sees that T2's absolute deadline 7 is earlier than the newly released T1 deadline 10, so it completes T2 first. Later, it similarly lets the second T2 job with deadline 14 continue through the T1 release at time 10.

When deadlines differ from periods, tasks share locks, execution is non-preemptible, or multiple CPUs are involved, these simple tests no longer apply directly. Blocking terms, release jitter, migration, and multiprocessor interference require richer analysis. Linux `SCHED_DEADLINE` uses Global EDF with Constant Bandwidth Server concepts and requires `runtime <= deadline <= period`; the kernel also performs admission control, as documented in [`sched(7)`](https://man7.org/linux/man-pages/man7/sched.7.html) and the [deadline scheduler documentation](https://docs.kernel.org/scheduler/sched-deadline.html).

### **Linux Scheduling: From CFS to EEVDF**

Linux exposes several scheduling policies rather than forcing every workload through one queueing rule. User-visible normal policies include `SCHED_OTHER`, `SCHED_BATCH`, and `SCHED_IDLE`; real-time policies include `SCHED_FIFO` and `SCHED_RR`; `SCHED_DEADLINE` accepts explicit runtime, deadline, and period parameters. These policies have different privilege requirements and semantics. A runaway FIFO or deadline thread can deny CPU service to ordinary work, so experiments require resource limits and a recovery path.

The **Completely Fair Scheduler** (CFS) established the key mental model for normal tasks: approximate an ideal processor that continuously divides service among runnable tasks. Each task accumulates **virtual runtime**. A task with greater weight accumulates virtual runtime more slowly for the same real CPU time, which gives it a larger long-run share. Selecting the runnable task with the smallest virtual runtime favors the task that is furthest behind its weighted entitlement.

Linux began transitioning the fair scheduling class toward **Earliest Eligible Virtual Deadline First** (EEVDF) in kernel 6.6. EEVDF keeps virtual-time fairness accounting but separates two questions:

1. **Eligibility:** lag measures the difference between entitled and received service. A positive lag means the task is owed CPU time; a negative lag means it has received more than its share. Only eligible tasks, described by the kernel documentation as lag greater than or equal to zero, compete in the next step.
2. **Latency choice:** among eligible tasks, choose the earliest virtual deadline. Shorter requested slices can therefore express latency sensitivity without discarding proportional fairness.

![CFS selects the smallest weighted virtual runtime, while EEVDF first filters by lag and then selects the earliest virtual deadline.](assets/cfs-eevdf-concepts.svg){fig-alt="Conceptual two-panel comparison of CFS virtual runtime and EEVDF eligibility plus virtual deadline." width="98%"}

*Figure: original conceptual diagram based on the official Linux [CFS design](https://docs.kernel.org/scheduler/sched-design-CFS.html) and [EEVDF scheduler](https://docs.kernel.org/scheduler/sched-eevdf.html) documentation.*

The figure is intentionally conceptual. Linux also handles waking sleepers, per-CPU queues, cgroup hierarchy, bandwidth control, affinity, migration, NUMA, and architecture-specific capacity. Internal data structures and tunables evolve; the stable lesson is that fairness accounting and latency selection are related but distinct.

`nice` values are not absolute reservations. They influence relative weight among competing normal tasks. Likewise, cgroup `cpu.weight` expresses relative share under contention, while bandwidth controls such as `cpu.max` can impose a quota. If other groups are idle, a weighted group may use more than its nominal fraction unless a quota limits it.

<details>
<summary><strong>Linux: inspect scheduling policy and observed execution</strong></summary>

```bash
# Per-thread class, real-time priority, nice value, last CPU, and state.
ps -eLo pid,tid,cls,rtprio,ni,pri,psr,stat,comm --sort=pid,tid

# Read the current shell's policy and priority.
chrt -p $$

# Inspect scheduler accounting exported for one process.
cat /proc/$$/sched

# Record and display a short scheduler timeline when perf is available.
sudo perf sched record -- sleep 2
sudo perf sched timehist
```

Field availability and permissions depend on the kernel and distribution. `/proc/PID/sched` is an implementation-oriented diagnostic interface rather than a portable application API.

</details>

For the `cat | grep` pipeline, normal fair scheduling usually rewards the stages' actual runnable behavior rather than reserving two full CPUs. A stage blocked on the pipe is not competing for CPU. When a wakeup makes it runnable, latency policy affects how quickly the pipeline restarts. If many unrelated CPU-bound tasks compete, group weights and placement may matter more than the names or process boundaries of `cat` and `grep`.

### **Simulating and Comparing Scheduling Policies**

A scheduling simulator makes assumptions executable. It should separate the workload from the policy, log every execution interval, and derive metrics from the log rather than from hand-maintained counters. The compact simulator below implements FCFS, non-preemptive SJF, unit-resolution SRTF, and Round Robin for integer arrival and burst times.

<details>
<summary><strong>Python: discrete-event comparison of FCFS, SJF, SRTF, and Round Robin</strong></summary>

```python
from collections import deque
from dataclasses import dataclass
from statistics import mean


@dataclass(frozen=True)
class Job:
    name: str
    arrival: int
    burst: int


JOBS = [
    Job("A", arrival=0, burst=8),
    Job("B", arrival=1, burst=4),
    Job("C", arrival=2, burst=2),
    Job("D", arrival=4, burst=1),
]


def append_segment(timeline, name, start, end):
    """Append an interval, merging adjacent runs of the same job."""
    if timeline and timeline[-1][0] == name and timeline[-1][2] == start:
        old_name, old_start, _ = timeline[-1]
        timeline[-1] = (old_name, old_start, end)
    else:
        timeline.append((name, start, end))


def run_non_preemptive(jobs, policy):
    """Run FCFS or SJF; a selected job keeps the CPU until completion."""
    pending = list(jobs)
    timeline = []
    now = min(job.arrival for job in pending)

    while pending:
        ready = [job for job in pending if job.arrival <= now]
        if not ready:
            # No runnable job: move simulated time to the next arrival.
            now = min(job.arrival for job in pending)
            ready = [job for job in pending if job.arrival <= now]

        if policy == "FCFS":
            job = min(ready, key=lambda item: (item.arrival, item.name))
        elif policy == "SJF":
            job = min(ready, key=lambda item: (item.burst, item.arrival, item.name))
        else:
            raise ValueError(f"unknown non-preemptive policy: {policy}")

        append_segment(timeline, job.name, now, now + job.burst)
        now += job.burst
        pending.remove(job)

    return timeline


def run_srtf(jobs):
    """Reconsider the shortest remaining job at each integer time unit."""
    remaining = {job.name: job.burst for job in jobs}
    by_name = {job.name: job for job in jobs}
    timeline = []
    now = min(job.arrival for job in jobs)

    while any(value > 0 for value in remaining.values()):
        ready = [
            job for job in jobs
            if job.arrival <= now and remaining[job.name] > 0
        ]
        if not ready:
            now = min(
                job.arrival for job in jobs
                if remaining[job.name] > 0 and job.arrival > now
            )
            continue

        job = min(
            ready,
            key=lambda item: (remaining[item.name], item.arrival, item.name),
        )
        append_segment(timeline, job.name, now, now + 1)
        remaining[job.name] -= 1
        now += 1

    return timeline


def run_round_robin(jobs, quantum):
    """Add arrivals during a slice before requeueing the expired job."""
    pending = deque(sorted(jobs, key=lambda item: (item.arrival, item.name)))
    ready = deque()
    remaining = {job.name: job.burst for job in jobs}
    timeline = []
    now = pending[0].arrival

    while pending or ready:
        while pending and pending[0].arrival <= now:
            ready.append(pending.popleft())

        if not ready:
            now = pending[0].arrival
            continue

        job = ready.popleft()
        duration = min(quantum, remaining[job.name])
        append_segment(timeline, job.name, now, now + duration)
        now += duration
        remaining[job.name] -= duration

        # Arrivals at the boundary enter before the expired job is requeued.
        while pending and pending[0].arrival <= now:
            ready.append(pending.popleft())
        if remaining[job.name] > 0:
            ready.append(job)

    return timeline


def calculate_metrics(jobs, timeline):
    """Derive first start and completion from the execution trace."""
    first_start = {}
    completion = {}
    for name, start, end in timeline:
        first_start.setdefault(name, start)
        completion[name] = end

    rows = []
    for job in jobs:
        turnaround = completion[job.name] - job.arrival
        waiting = turnaround - job.burst
        response = first_start[job.name] - job.arrival
        rows.append((job.name, waiting, response, turnaround))
    return rows


def print_report(name, jobs, timeline):
    rows = calculate_metrics(jobs, timeline)
    gantt = " | ".join(f"{job} {start}-{end}" for job, start, end in timeline)
    print(f"{name:4}  {gantt}")
    print(
        "      averages: "
        f"wait={mean(row[1] for row in rows):.2f}, "
        f"response={mean(row[2] for row in rows):.2f}, "
        f"turnaround={mean(row[3] for row in rows):.2f}"
    )


experiments = {
    "FCFS": run_non_preemptive(JOBS, "FCFS"),
    "SJF": run_non_preemptive(JOBS, "SJF"),
    "SRTF": run_srtf(JOBS),
    "RR": run_round_robin(JOBS, quantum=2),
}

for policy_name, execution_trace in experiments.items():
    print_report(policy_name, JOBS, execution_trace)
```

Expected output:

```text
FCFS  A 0-8 | B 8-12 | C 12-14 | D 14-15
      averages: wait=6.75, response=6.75, turnaround=10.50
SJF   A 0-8 | D 8-9 | C 9-11 | B 11-15
      averages: wait=5.25, response=5.25, turnaround=9.00
SRTF  A 0-1 | B 1-2 | C 2-4 | D 4-5 | B 5-8 | A 8-15
      averages: wait=2.50, response=0.00, turnaround=6.25
RR    A 0-2 | B 2-4 | C 4-6 | A 6-8 | D 8-9 | B 9-11 | A 11-15
      averages: wait=4.75, response=1.75, turnaround=8.50
```

</details>

The simulator intentionally omits context-switch cost, I/O blocking, multiple CPUs, prediction error, and cache effects. That is a feature if the goal is to isolate policy, but it is a limitation if the results are presented as operating-system performance. A stronger experiment should:

1. define arrival and burst distributions or replay a trace;
2. include switch and migration cost;
3. run long enough to separate warm-up from steady state;
4. report per-class and tail metrics, not only global means;
5. repeat randomized policies with fixed recorded seeds and confidence intervals;
6. validate the model against measured scheduler traces before making system claims.

For Linux observations, `perf sched timehist`, tracepoints, and per-thread `/proc` accounting can reveal wakeup-to-run delay, runtime, migrations, and switch sequences. Measurements should be collected under controlled affinity, frequency, and background load. The simulator answers "what does this policy imply under this model?"; tracing answers "what did this implementation do on this machine?"

### **Comparison and Summary**

| Policy | Preemptive? | Information or state required | Main strength | Main risk |
|---|---|---|---|---|
| FCFS | no | arrival order | minimal overhead and transparent order | convoy effect and poor short-job response |
| SJF | no | predicted full burst | minimum mean waiting under ideal assumptions | prediction error, starvation, no reaction mid-burst |
| SRTF | yes | predicted remaining time | excellent mean flow time for known sizes | starvation and frequent preemption |
| Round Robin | yes | FIFO queue and quantum | bounded rotation among peers, good initial response | quantum overhead and no group fairness by itself |
| strict priority | either | trusted priority | protects explicitly urgent classes | starvation and priority inversion dependencies |
| MLFQ | yes | queues plus usage history | adapts to interactive and CPU-bound behavior | tunable complexity and gaming without robust accounting |
| lottery | normally | tickets plus randomness | flexible proportional shares | short-term variance |
| stride | normally | tickets, stride, pass | deterministic proportional shares | dynamic membership and normalization complexity |
| RMS | yes | period and WCET model | analyzable fixed priorities | conservative bound and strict assumptions |
| EDF | yes | absolute deadlines and budgets | strong uniprocessor utilization result | overload and implementation require careful control |
| CFS/EEVDF family | yes | weights, virtual time, lag, virtual deadlines | practical weighted fairness with latency awareness | behavior also depends on hierarchy, placement, and kernel details |

A reasonable selection process starts from the contract:

- use FCFS as a simple baseline or where non-preemptive order is required;
- use shortest-job reasoning when sizes are known or predictably estimated and mean completion time dominates;
- use Round Robin or feedback queues for responsive time sharing without exact size knowledge;
- use hierarchical proportional share when tenants or groups need relative service isolation;
- use a real-time class only with valid timing parameters, bounded interference, admission control, and operational safeguards;
- on multicore systems, evaluate placement and memory locality together with temporal policy.

Several misconceptions are now easier to reject. A runnable thread is not the same as a running thread. A timer interrupt need not cause a switch. A smaller quantum is not free responsiveness. Equal per-thread treatment is not necessarily equal per-user treatment. CPU affinity is not CPU reservation. High average utilization is not proof of good latency, and a high priority is not a deadline guarantee.

The durable model is a loop: events change eligibility, accounting records received service, policy ranks legal candidates, placement chooses a processor, and dispatch realizes the decision. The quality of a scheduler is the quality of that loop under stated workload assumptions, measured against an explicit combination of response, throughput, fairness, locality, predictability, and deadlines.
